> verl 和 slime 默认都不是从“正在训练的 actor 对象”实时复制出 critic，而是让 critic 独立加载同一个基础 checkpoint 的 Transformer 主干，再把语言模型的词表输出头替换为标量 value head。主干初值通常相同，但从初始化完成开始，actor/critic 就是两套独立参数、optimizer 和 checkpoint 状态。

- 设 Transformer 输出为 $h_t\in\mathbb{R}^{H}$，critic 为：$V_\psi(s_t)=w_v^\top h_t+b_v$ 其中 $w_v\in\mathbb{R}^{H}$。
    - 重点是：原来的 LM head 是 $W_{\text{lm}}\in\mathbb{R}^{|\mathcal V|\times H}$，它不会被“压缩”成 $w_v$；新 value head 通常需要随机初始化。

### verl ppo critic

- critic 主干从 critic.model.path 独立加载

```python
load_valuehead_model(
    local_path=critic.model.path,
    model_config=hf_config,
)
```
- 优先调用 `AutoModelForTokenClassification.from_pretrained(..., num_labels=1)`。

### slime ppo-critic

> 明确替换为 $H\to1$ 输出层

slime 创建普通 Megatron GPTModel 后，如果 role == "critic"：

```
model.output_layer = LinearForLastLayer(
    input_size=config.hidden_size,
    output_size=1,
    config=config,
)
```